In [174]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from xgboost import XGBClassifier

# Ignoro i warning 
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

# Percorso da cui prendere il file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Carico il csv e stampo la shape

In [175]:
df_duke = pd.read_csv(FILE_PATH / "duke_lesions.csv")

print("DUKE shape: ", df_duke.shape)

DUKE shape:  (291, 109)


# Target

In [176]:
marker = ["PR", "ER"]

# Bilancio il dataset globalmente

In [177]:
balanced_datasets = {}

for m in marker:
    print(f"\nBilanciamento classi per {m}")

    # Separo le classi
    df_0 = df_duke[df_duke[m] == 0]
    df_1 = df_duke[df_duke[m] == 1]

    # Controllo di sicurezza
    if len(df_0) == 0 or len(df_1) == 0:
        raise ValueError(f"{m}: una delle classi è vuota")

    # Dimensione della classe minoritaria
    n_min = min(len(df_0), len(df_1))
    print("Dimensione per classe:", n_min)

    # Undersampling GLOBALE
    df_balanced = pd.concat([
        df_0.sample(n=n_min, random_state=42),
        df_1.sample(n=n_min, random_state=42)
    ]).sample(frac=1, random_state=42).reset_index(drop=True)

    balanced_datasets[m] = df_balanced


Bilanciamento classi per PR
Dimensione per classe: 134

Bilanciamento classi per ER
Dimensione per classe: 123


# Controllo il numero delle classi

In [178]:
for m in marker:
    vc = balanced_datasets[m][m].value_counts(dropna=False)

    print(f"\nDistribuzione {m} – DUKE")
    print("-" * 30)
    print("Negativi:", vc.get(0, 0))
    print("Positivi:", vc.get(1, 0))



Distribuzione PR – DUKE
------------------------------
Negativi: 134
Positivi: 134

Distribuzione ER – DUKE
------------------------------
Negativi: 123
Positivi: 123


# Stratified Cross-Validation

In [179]:
FEATURES = [c for c in df_duke.columns if c.startswith("original_")]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Training

In [180]:
results = {}

for target in marker:
    print(f"\n==============================")
    print(f" Training e CV per {target}")
    print(f"==============================")

    df_bal = balanced_datasets[target]

    X = df_bal[FEATURES]
    y = df_bal[target]


    acc_scores = []
    f1_scores = []
    auc_scores = []
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):

        # Split
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Bilanciamento solo in fase di  training
        X_train_bal, y_train_bal = X_train, y_train

        print(f"\n[DEBUG] {target} – Fold {fold}")
        print("X_train_bal shape:", X_train_bal.shape)
        print("y_train_bal shape:", y_train_bal.shape)

        print("Distribuzione classi bilanciate: ", y_train_bal.value_counts())


        # Definisco il modello
        model = XGBClassifier(
            random_state=42,
            n_jobs=1,
            objective='binary:logistic',
            eval_metric='logloss',
            tree_method='hist',
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=3,
            reg_alpha=0,
            reg_lambda=1,
            scale_pos_weight=1
        )

        model.fit(X_train_bal, y_train_bal)

        # Predizioni
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        # Metriche
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)

        acc_scores.append(acc)
        f1_scores.append(f1)
        auc_scores.append(auc)

        print(
            f"Fold {fold} | "
            f"ACC={acc:.3f} | "
            f"F1={f1:.3f} | "
            f"AUC={auc:.3f}"
        )
    # Media ± std
    results[target] = {
        "Accuracy": (np.mean(acc_scores), np.std(acc_scores)),
        "F1": (np.mean(f1_scores), np.std(f1_scores)),
        "AUC": (np.mean(auc_scores), np.std(auc_scores)),
    }

    print("\n--- RISULTATI FINALI ---")
    print(f"Accuracy : {results[target]['Accuracy'][0]:.3f} ± {results[target]['Accuracy'][1]:.3f}")
    print(f"F1-score : {results[target]['F1'][0]:.3f} ± {results[target]['F1'][1]:.3f}")
    print(f"ROC-AUC  : {results[target]['AUC'][0]:.3f} ± {results[target]['AUC'][1]:.3f}")





 Training e CV per PR

[DEBUG] PR – Fold 1
X_train_bal shape: (214, 105)
y_train_bal shape: (214,)
Distribuzione classi bilanciate:  PR
0    107
1    107
Name: count, dtype: int64
Fold 1 | ACC=0.556 | F1=0.520 | AUC=0.487

[DEBUG] PR – Fold 2
X_train_bal shape: (214, 105)
y_train_bal shape: (214,)
Distribuzione classi bilanciate:  PR
0    107
1    107
Name: count, dtype: int64
Fold 2 | ACC=0.574 | F1=0.623 | AUC=0.583

[DEBUG] PR – Fold 3
X_train_bal shape: (214, 105)
y_train_bal shape: (214,)
Distribuzione classi bilanciate:  PR
0    107
1    107
Name: count, dtype: int64
Fold 3 | ACC=0.630 | F1=0.630 | AUC=0.612

[DEBUG] PR – Fold 4
X_train_bal shape: (215, 105)
y_train_bal shape: (215,)
Distribuzione classi bilanciate:  PR
1    108
0    107
Name: count, dtype: int64
Fold 4 | ACC=0.566 | F1=0.596 | AUC=0.561

[DEBUG] PR – Fold 5
X_train_bal shape: (215, 105)
y_train_bal shape: (215,)
Distribuzione classi bilanciate:  PR
0    108
1    107
Name: count, dtype: int64
Fold 5 | ACC=0.491 